### Tools

#### Models can request to call tools that perform tasks such as fetching data from a dataset, searching the web, or running code. Tools are pairings of: 
1. A schema, including the name of the tool, a description, and/or argument definitions (ofter a JSON schema)
2. A function or coroutine to execute.

In [29]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=ChatGroq(model="qwen/qwen3-32b")
response=model.invoke("Hello")
response.content


'<think>\nOkay, the user said "Hello". I should respond in a friendly and welcoming way. Let me make sure to use an emoji to keep it approachable. Maybe something like a smiley face. I\'ll ask them how I can assist today. Keep it simple and open-ended.\n</think>\n\nHello! 😊 How can I assist you today?'

In [30]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
from langchain.agents import create_agent

def get_weather(city:str)->str:
    """Get the current weather for a city."""
    return f"The weather in {city} is sunny."

agent=create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)
agent
response=agent.invoke({"messages":[{"role":"user", "content": "What is the weather in New York?"}]})
response["messages"][-1] # For seeing last message in the response.

AIMessage(content='The weather in New York is sunny.', additional_kwargs={'reasoning_content': "Okay, the user asked about the weather in New York. I called the get_weather function with the city set to New York. The response came back saying it's sunny. Now I need to relay that information clearly. Let me check if there's any additional details I should mention, but the function only provided the weather condition. So I'll just state that the weather is sunny in New York.\n"}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 202, 'total_tokens': 295, 'completion_time': 0.162036616, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.009136651, 'prompt_tokens_details': None, 'queue_time': 0.160619068, 'total_time': 0.171173267}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e5416-e1a8-7851-8997-bf83995

In [31]:
from langchain.tools import tool
@tool

def get_weather(location: str)->str:
	"""Get the current weather of the location"""
	return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])
response=model_with_tools.invoke("What is the current weather of Dhaka?")

print(response)

for tool_call in response.tool_calls:
	print(f"Tool:{tool_call['name']}")
	print(f"Args:{tool_call['args']}")


content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the current weather in Dhaka. I need to use the get_weather function. The function requires a location parameter, which in this case is Dhaka. So I\'ll call the function with "Dhaka" as the location. Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'nznz35rmr', 'function': {'arguments': '{"location":"Dhaka"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 156, 'total_tokens': 259, 'completion_time': 0.153682375, 'completion_tokens_details': {'reasoning_tokens': 77}, 'prompt_time': 0.006388824, 'prompt_tokens_details': None, 'queue_time': 0.054367996, 'total_time': 0.160071199}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id=

### Tool Execution Loops

In [32]:
#step 1: Model generates tool calls
messages=[{"role":"user", "content": "What is the weather in New York?"}]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

# step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)
# step 3: Pass results back to model for final response
final_response=model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in New York is sunny."

<answer>
The weather in New York is sunny. It's a great day to enjoy outdoor activities!
</answer>


In [33]:
messages

[{'role': 'user', 'content': 'What is the weather in New York?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in New York. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. The user provided "New York" as the location. So I should call get_weather with location set to "New York". I need to make sure the JSON is correctly formatted. Let me structure the tool_call accordingly.\n', 'tool_calls': [{'id': 'j3zq2aqf4', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 110, 'prompt_tokens': 155, 'total_tokens': 265, 'completion_time': 0.169617657, 'completion_tokens_details': {'reasoning_tokens': 85}, 'prompt_time': 0.007769547, 'prompt_tokens_details': None, 'queue_time': 0.160347782, 'total_time': 0.177387204}, 'model_name': 'qwen/qwen3-3